# Reproducing the LAMG+ vs. approxChol comparison

This notebook **re-runs the two robust near-linear solvers live** — **LAMG+** (this package) and
**approxChol** (`approxchol_lap`, Laplacians.jl) — on several graphs taken directly from the paper's
multi-solver comparison table, and checks that the measured timing and accuracy match the reported
values.

* **Metric.** Setup\\+solve wall-clock to relative residual $\le 10^{-8}$, reported as **microseconds per
  nonzero** of $L$: $(\text{setup}+\text{solve sec})\cdot10^6 / \mathrm{nnz}(L)$ with $\mathrm{nnz}(L)=n+2m$.
  This is the paper's normalization (verified to reproduce the table to two digits).
* **What reproduces exactly vs. approximately.** Wall-clock varies run-to-run and across machines (the
  paper used an Apple M5 Pro), so the μs/nnz values agree only **within timing variance**. The
  **winner** (which solver is faster) and the **accuracy** (final residual, cycle/PCG-iteration counts)
  reproduce deterministically.
* **Same code path as the paper.** Loader, largest-connected-component reduction, fixed zero-sum RHS
  $b=L\,x_{\text{true}}$, and the solver wrappers live in `scripts/repro_lib.jl`, extracted verbatim from
  `scripts/competitor_benchmark.jl`.

Run from the `examples/` directory (the cells use paths relative to it).

In [1]:
using Pkg
Pkg.activate("repro_env")              # lean env: LAMG (dev) + Laplacians.jl
include("../scripts/repro_lib.jl")     # read_mm_adj, reduce_to_lcc, bench_*, repro_graph
println("environment ready")

  Activating 

environment ready


project at `~/code/mg/lamgplus/examples/repro_env`


## Graphs and reported values

Four entries from the comparison table: two web graphs (approxChol wins) and two
finite-element/structural matrices (LAMG+ wins) — the clean class split. `DATA_DIR` points to the
SuiteSparse `.mtx` files; download them from https://sparse.tamu.edu if reproducing elsewhere.

In [2]:
# Put the four SuiteSparse .mtx files in ../data (repo convention), or set $LAMGPLUS_DATA.
DATA_DIR = get(ENV, "LAMGPLUS_DATA", "../data")

# (name substring, reported LAMG+ μs/nnz, reported approxChol μs/nnz, paper winner)
TABLE = [
    ("web-Stanford", 0.26, 0.19, "approxChol"),
    ("web-Google",   0.48, 0.25, "approxChol"),
    ("bmwcra_1",     0.08, 0.28, "LAMG+"),
    ("pwtk",         0.11, 0.21, "LAMG+"),
]
length(TABLE)

4

## Run both solvers on each graph (live)

In [3]:
using Printf
results = NamedTuple[]
for (g, rep_lp, rep_apx, rep_win) in TABLE
    file = only(filter(f -> occursin(g, f), readdir(DATA_DIR; join=true)))
    @printf("running %-14s ... ", g); flush(stdout)
    r = repro_graph(file; tol=1e-8, repeats=3)
    push!(results, (g=g, r=r, rep_lp=rep_lp, rep_apx=rep_apx, rep_win=rep_win))
    @printf("n=%-9d m=%-9d  done\n", r.n, r.m); flush(stdout)
end

running web-Stanford   ... 

n=213453    m=1045015    done


running web-Google     ... 

n=723783    m=2587555    done


running bmwcra_1       ... 

n=148770    m=5246416    done


running pwtk           ... 

n=217918    m=5653257    done


## Reproduced vs. reported

`repro` = this live run; `paper` = the value in the comparison table.

In [4]:
@printf("%-13s | %-13s | %-13s | %-21s | %-15s\n",
        "graph", "LAMG+ μs/nnz", "approxChol", "winner", "accuracy (LAMG+/AC)")
@printf("%-13s | %6s %6s | %6s %6s | %-10s %-10s | %s\n",
        "", "repro","paper","repro","paper","repro","paper","resid OK?")
println("-"^96)
for x in results
    r = x.r
    win_ok = r.winner == x.rep_win
    acc_ok = r.lamg_ok && r.apx_ok
    @printf("%-13s | %6.2f %6.2f | %6.2f %6.2f | %-10s %-10s | %s (%dcyc %.0e / %dit %.0e)\n",
            x.g, r.lamg_us_per_nnz, x.rep_lp, r.apx_us_per_nnz, x.rep_apx,
            r.winner, x.rep_win, (win_ok&&acc_ok) ? "yes" : "NO",
            r.lamg_cycles, r.lamg_rel, r.apx_iters, r.apx_rel)
end

graph         | LAMG+ μs/nnz  | approxChol    | winner                | accuracy (LAMG+/AC)
              |  repro  paper |  repro  paper | repro      paper      | resid OK?


------------------------------------------------------------------------------------------------
web-Stanford  |   0.30   0.26 |   0.24   0.19 | approxChol approxChol | yes (4cyc 7e-09 / 56it 6e-09)
web-Google    |   0.51   0.48 |   0.28   0.25 | approxChol approxChol | yes (4cyc 7e-09 / 31it 9e-09)


bmwcra_1      |   0.09   0.08 |   0.25   0.28 | LAMG+      LAMG+      | yes (5cyc 6e-10 / 13it 7e-09)
pwtk          |   0.10   0.11 |   0.24   0.21 | LAMG+      LAMG+      | yes (5cyc 3e-09 / 17it 9e-09)


## Checks: winners and convergence reproduce exactly

In [5]:
# Deterministic guarantees (assert): the faster solver and the convergence reproduce exactly.
@assert all(x -> x.r.winner == x.rep_win, results)   "a winner disagrees with the paper"
@assert all(x -> x.r.lamg_ok && x.r.apx_ok, results) "a solve failed to reach 1e-8"
# Wall-clock is not deterministic; report how close the per-nonzero times landed.
dev(a, b) = max(a / b, b / a)
worst = maximum(max(dev(x.r.lamg_us_per_nnz, x.rep_lp), dev(x.r.apx_us_per_nnz, x.rep_apx)) for x in results)
println("✓ every winner matches the paper")
println("✓ every solve reached the 1e-8 tolerance")
@printf("• per-nonzero times match the table within %.2f× (run-to-run wall-clock variance)\n", worst)

✓ every winner matches the paper
✓ every solve reached the 1e-8 tolerance
• per-nonzero times match the table within 1.26× (run-to-run wall-clock variance)


## Conclusion

The live re-run reproduces the reported comparison: LAMG+ is faster on the finite-element/structural
matrices (`bmwcra_1`, `pwtk`) and approxChol on the web graphs (`web-Stanford`, `web-Google`), the
**class split** the paper reports. Per-nonzero times agree with the table within timing variance, and
both solvers reach the $10^{-8}$ tolerance in the expected handful of cycles / PCG iterations.

To reproduce the **full** $201$-graph table (and the other five solvers), run
`scripts/competitor_benchmark.jl` over `data/competitor_instances.txt`; this notebook is the fast,
self-contained spot check.